# 复现 When Thinking Fails：Llama3-8B-Instruct 在 IFEval 上的 DA vs CoT

- **模型**：`../models/llama3-8B-Instruct`（Meta-Llama-3-8B-Instruct，HuggingFace repo `meta-llama/Meta-Llama-3-8B-Instruct`）
- **数据**：`../ifeval/train.jsonl`（541 条 IFEval 指令）
- **框架**：vLLM BF16，`gpu_memory_utilization=0.85`，RTX 4090 24GB
- **采样**：论文明确只给了 `temperature=0`；`max_tokens=8192`、`max_model_len=16384`、`top_p=0.95` 沿用项目惯例

## 两个条件

| 条件 | 提示词 | 期望 |
|---|---|---|
| **DA**（direct answer） | IFEval instruction 直接作为 user 消息，走 Llama3 chat template | 直接输出答案，无思考 |
| **CoT** | 论文 Appendix F 英文 Reasoning Prompt，替换 `{question}` 为 IFEval instruction，走 chat template | 模型先写 `THINK:` 再写 `ANSWER:` |

## 两个判分口径

- **strict**：项目已验证的 25 种类型逐条判定器
- **loose**：在 strict 基础上额外忽略 markdown 包裹、宽松归一化（近似论文的 instruction-level loose accuracy）

In [ ]:
# ===== 全局配置 =====
import os, re, json, time, ast
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

# —— 路径 ——
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
MODEL_PATH = ROOT / "models" / "llama3-8B-Instruct"
DATA_PATH = ROOT / "ifeval" / "train.jsonl"
RESULT_DIR = Path.cwd() / "ifeval_llama3_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# —— 采样（论文只明确给 temperature=0，其余沿用项目惯例）——
TEMPERATURE = 0
TOP_P = 0.95
SEED = 0
MAX_TOKENS = 8192
MAX_MODEL_LEN = 16384

# —— vLLM 必须在 import 前关掉 FlashInfer 采样器 ——
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

print(f"模型: {MODEL_PATH}")
print(f"数据: {DATA_PATH}")
print(f"结果: {RESULT_DIR}")
print(f"采样: temp={TEMPERATURE}, top_p={TOP_P}, max_tokens={MAX_TOKENS}")

## 1. 加载 IFEval

从 gitignore 的本地 JSONL 读入，每行是 huggingface datasets 导出的一条记录。

In [ ]:
# ===== 读 IFEval JSONL =====
def _read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def _as_list(x):
    """兼容 ndarray / list / dict / JSON 字符串等各种形态。"""
    if x is None: return []
    if isinstance(x, dict): return [x]
    if isinstance(x, str):
        s = x.strip()
        if len(s) >= 2 and s[0] in "[(" and s[-1] in "])":
            try:
                v = ast.literal_eval(s)
                if isinstance(v, (list, tuple)): return list(v)
            except Exception:
                pass
        return [x]
    try: return list(x)
    except TypeError: return [x]


raw = _read_jsonl(DATA_PATH)
df = pd.DataFrame(raw).reset_index(drop=True)
df["uid"] = [f"ifeval-{i:04d}" for i in range(len(df))]
df["instruction_ids"] = df["instruction_id_list"].map(lambda x: [str(s) for s in _as_list(x)])
df["instruction_kwargs"] = df["kwargs"].map(lambda x: [d if isinstance(d, dict) else {} for d in _as_list(x)])
df["k"] = df["instruction_ids"].map(len)

print(f"共 {len(df)} 条")
kdist = df["k"].value_counts().sort_index().rename_axis("k").reset_index(name="count")
print(kdist.to_string(index=False))

## 2. DA / CoT 提示词构造

- **DA**：把 IFEval 的原始 instruction 直接作为 user 消息送进 chat template，不加任何 system prompt。
- **CoT**：把论文 Appendix F 的英文 Reasoning Prompt 作为 user 消息，`{question}` 替换为 IFEval instruction，再走 chat template。

Llama3-8B-Instruct 不是 thinking 模型，它没有内建的 `<think>...</think>` 协议，所以 CoT 条件完全靠显式的英文 prompt 引导模型输出 `THINK:` 与 `ANSWER:` 两段。

In [ ]:
# ===== DA 条件：IFEval instruction 直接作 user 消息 =====
def messages_da(instruction):
    return [{"role": "user", "content": instruction}]


# ===== CoT 条件：论文 Appendix F 英文 Reasoning Prompt =====
# 原文格式（PDF 抽取后复原换行）：
# You will be given an instruction. You need to first carefully
# think step by step about the constraints and conditions in
# the instruction and then provide the answer.
# Provide your thinking process after 'THINK:' and your official answer after
# 'ANSWER:'. The given instruction is the following:
# INSTRUCTION:
# {question}
# Now, generate your thinking process and final answer for the
# given instruction.
COT_REASONING_PROMPT = """You will be given an instruction. You need to first carefully
think step by step about the constraints and conditions in
the instruction and then provide the answer.
Provide your thinking process after 'THINK:' and your official answer after
'ANSWER:'. The given instruction is the following:
INSTRUCTION:
{question}
Now, generate your thinking process and final answer for the
given instruction."""


def messages_cot(instruction):
    prompt = COT_REASONING_PROMPT.format(question=instruction)
    return [{"role": "user", "content": prompt}]


# ===== CoT 输出切分：从 THINK: 与 ANSWER: 标签中抽出 THINK 段和 ANSWER 段 =====
def split_cot_output(text):
    """
    返回 (think_text, answer_text)。模型可能不严格遵守标签格式：
    - 两种标签都有 → 正常切分
    - 只有 THINK → THINK 之后的全部当 ANSWER
    - 只有 ANSWER → 空 THINK，ANSWER 之后的当回答
    - 都没有 → 空 THINK，全文当回答
    """
    s = str(text or "").strip()
    think = ""
    answer = s
    m_think = re.search(r"THINK:\s*", s)
    m_answer = re.search(r"ANSWER:\s*", s)
    if m_think and m_answer:
        think = s[m_think.end():m_answer.start()].strip()
        answer = s[m_answer.end():].strip()
    elif m_think:
        think = s[m_think.end():].strip()
        answer = ""
    elif m_answer:
        think = ""
        answer = s[m_answer.end():].strip()
    return think, answer


# 小预览
_sample_instr = df.iloc[0]["prompt"]
_print_n = 120
print("=== DA user message 预览（截断）===\n")
print(messages_da(_sample_instr)["content"][:_print_n])
print("\n=== CoT user message 预览（截断）===\n")
print(messages_cot(_sample_instr)["content"][:_print_n])

## 3. vLLM 加载与推理

两次独立推理：DA 541 条、CoT 541 条。temperature=0 是论文明确报告的唯一采样参数；其余沿用项目惯例。

In [ ]:
# ===== 加载 vLLM =====
from vllm import LLM, SamplingParams

llm = LLM(
    model=str(MODEL_PATH),
    dtype="bfloat16",
    gpu_memory_utilization=0.85,
    max_model_len=MAX_MODEL_LEN,
)
tokenizer = llm.get_tokenizer()

sp = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS,
    seed=SEED,
)


# ===== 预渲染所有 prompt =====
da_prompts = [
    tokenizer.apply_chat_template(messages_da(p), tokenize=False, add_generation_prompt=True)
    for p in df["prompt"]
]
cot_prompts = [
    tokenizer.apply_chat_template(messages_cot(p), tokenize=False, add_generation_prompt=True)
    for p in df["prompt"]
]


# ===== DA 推理 =====
print("=== 开始 DA 推理 ===")
t0 = time.time()
da_outputs = llm.generate(da_prompts, sp)
print(f"DA 完成，用时 {time.time() - t0:.1f}s")


# ===== CoT 推理 =====
print("=== 开始 CoT 推理 ===")
t0 = time.time()
cot_outputs = llm.generate(cot_prompts, sp)
print(f"CoT 完成，用时 {time.time() - t0:.1f}s")


# ===== 整理成 records =====
def _collect(outputs, condition):
    rows = []
    for i, (_, row) in enumerate(df.iterrows()):
        raw = outputs[i].outputs[0].text
        if condition == "cot":
            think, answer = split_cot_output(raw)
        else:
            think, answer = "", raw
        rows.append({
            "uid": row["uid"],
            "k": int(row["k"]),
            "instruction_ids": row["instruction_ids"],
            "instruction_kwargs": row["instruction_kwargs"],
            "prompt": row["prompt"],
            "raw_output": raw,
            "think_segment": think,
            "answer_segment": answer,
            "n_tokens": len(outputs[i].outputs[0].token_ids),
            "finish_reason": str(outputs[i].outputs[0].finish_reason),
        })
    return rows

da_rows = _collect(da_outputs, "da")
cot_rows = _collect(cot_outputs, "cot")

# 落盘原始输出（判分在下一节做）
da_df = pd.DataFrame(da_rows)
cot_df = pd.DataFrame(cot_rows)
da_df.to_json(RESULT_DIR / "da_raw.jsonl", orient="records", lines=True, force_ascii=False)
cot_df.to_json(RESULT_DIR / "cot_raw.jsonl", orient="records", lines=True, force_ascii=False)
print(f"原始输出已保存：da_raw.jsonl, cot_raw.jsonl")

## 4. Strict 判分器

项目 `IFEval_test.ipynb` 已验证的 25 种类型逐条判定器，**原样复用**。
约定：未知 id 或异常 → `passed=None`（不计入统计，jsonl 里留 `error`）。

In [ ]:
# ===== Strict 判分器：文本工具 =====
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:[''\-][A-Za-z0-9]+)*")
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def _norm_ws(text): return re.sub(r"\s+", " ", str(text or "")).strip()
def _word_tokens(text): return WORD_RE.findall(str(text or ""))
def _count_words(text): return len(_word_tokens(text))


def _split_sentences(text):
    text = str(text or "").strip()
    if not text: return []
    try:
        import nltk
        return [s for s in nltk.sent_tokenize(text) if s.strip()]
    except Exception:
        pass
    return [s for s in SENT_SPLIT_RE.split(text) if s.strip()]


def _has_phrase(text, phrase):
    if phrase is None or str(phrase) == "": return False
    return re.search(r"(?<!\w)" + re.escape(str(phrase)) + r"(?!\w)",
                     str(text or ""), re.IGNORECASE) is not None


def _count_phrase(text, phrase):
    if phrase is None or str(phrase) == "": return 0
    return len(re.findall(r"(?<!\w)" + re.escape(str(phrase)) + r"(?!\w)",
                          str(text or ""), re.IGNORECASE))


def _compare(count, threshold, relation):
    rel = str(relation or "at least").strip().lower()
    if rel in ("at least", ">=", "no less than"): return count >= threshold
    if rel in ("less than", "<"): return count < threshold
    if rel in ("at most", "<=", "no more than"): return count <= threshold
    if rel in ("more than", ">"): return count > threshold
    if rel in ("exactly", "=="): return count == threshold
    raise ValueError(f"未知 relation: {relation!r}")


# ===== Strict 判分器：逐条指令函数 =====
def _chk_capital(text, kw): return bool(re.search(r"[A-Za-z]", text)) and re.search(r"[a-z]", text) is None
def _chk_lowercase(text, kw): return bool(re.search(r"[A-Za-z]", text)) and re.search(r"[A-Z]", text) is None


def _chk_capital_word_frequency(text, kw):
    n = sum(1 for w in _word_tokens(text) if w.isupper())
    return _compare(n, int(kw.get("capital_frequency", 0)), kw.get("capital_relation", "at least"))


def _chk_repeat_prompt(text, kw):
    prompt = _norm_ws(kw.get("prompt_to_repeat", "")).lower()
    return bool(prompt) and prompt in _norm_ws(text).lower()


def _chk_two_responses(text, kw):
    parts = str(text or "").split("******")
    return len(parts) == 2 and all(p.strip() for p in parts)


def _chk_highlighted_sections(text, kw):
    need = int(kw.get("num_highlights", 1))
    return len(re.findall(r"\*[^*\n]+\*", str(text or ""))) >= need


def _chk_number_placeholders(text, kw):
    need = int(kw.get("num_placeholders", 0) or 0)
    return len(re.findall(r"\[[^\[\]\n]*\]", str(text or ""))) >= need


def _chk_number_paragraphs(text, kw):
    need = int(kw.get("num_paragraphs", 0) or 0)
    parts = re.split(r"(?m)^\s*\*{3,}\s*$|\n\s*\n", str(text or ""))
    return len([p for p in parts if p.strip()]) == need


def _chk_multiple_sections(text, kw):
    splitter = str(kw.get("section_spliter") or "")
    need = int(kw.get("num_sections", 0) or 0)
    if not splitter: return None
    return str(text or "").count(splitter) >= need


def _chk_postscript(text, kw):
    marker = str(kw.get("postscript_marker", "P.S.")).lower()
    lines = [l.strip() for l in str(text or "").strip().splitlines() if l.strip()]
    return bool(lines) and lines[-1].lower().startswith(marker)


_CONSTRAINED_RESPONSE_OPTIONS = ("my answer is yes.", "my answer is no.", "my answer is maybe.")
def _chk_constrained_response(text, kw):
    low = _norm_ws(text).lower()
    return any(opt in low for opt in _CONSTRAINED_RESPONSE_OPTIONS)


def _chk_json_format(text, kw):
    s = str(text or "").strip()
    if s.startswith("```"):
        s = re.sub(r"^```[A-Za-z]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s).strip()
    try:
        json.loads(s); return True
    except Exception: pass
    for pat in (r"\{.*\}", r"\[.*\]"):
        m = re.search(pat, s, flags=re.DOTALL)
        if m:
            try: json.loads(m.group(0)); return True
            except Exception: continue
    return False


def _chk_number_bullet_lists(text, kw):
    need = int(kw.get("num_bullets", 1))
    bullets = re.findall(r"^\s*\*(?!\*)\s*\S.*$", str(text or ""), flags=re.MULTILINE)
    return len(bullets) == need


def _chk_title(text, kw): return re.search(r"<<[^\n<>]+>>", str(text or "")) is not None


def _chk_keywords_existence(text, kw):
    keywords = kw.get("keywords", [])
    if isinstance(keywords, str): keywords = [keywords]
    return all(_has_phrase(text, w) for w in keywords)


def _chk_keywords_frequency(text, kw):
    n = _count_phrase(text, kw.get("keyword", ""))
    return _compare(n, int(kw.get("frequency", 0)), kw.get("relation", "at least"))


def _chk_forbidden_words(text, kw):
    words = kw.get("forbidden_words", [])
    if isinstance(words, str): words = [words]
    return not any(_has_phrase(text, w) for w in words)


def _chk_letter_frequency(text, kw):
    letters = kw.get("letter", [])
    if isinstance(letters, str): letters = [letters]
    letters = {str(c).lower() for c in letters}
    n = sum(1 for c in str(text or "").lower() if c in letters)
    return _compare(n, int(kw.get("let_frequency", 0)), kw.get("let_relation", "at least"))


_SCRIPT_RE = {"zh": r"[\u4e00-\u9fff]", "ja": r"[\u3040-\u30ff\u4e00-\u9fff]",
              "ko": r"[\uac00-\ud7af]", "ru": r"[\u0400-\u04ff]",
              "ar": r"[\u0600-\u06ff]", "hi": r"[\u0900-\u097f]",
              "th": r"[\u0e00-\u0e7f]", "el": r"[\u0370-\u03ff]"}


def _chk_response_language(text, kw):
    lang = str(kw.get("language", "")).lower()
    s = str(text or "")
    try:
        from langdetect import detect
        return detect(s) == lang
    except Exception: pass
    if lang in _SCRIPT_RE:
        return re.search(_SCRIPT_RE[lang], s) is not None
    if lang in ("en", "english"):
        letters = [c for c in s if c.isalpha()]
        if not letters: return False
        return sum(ord(c) < 128 for c in letters) / len(letters) > 0.9
    return None


def _chk_number_sentences(text, kw):
    return _compare(len(_split_sentences(text)), int(kw.get("num_sentences", 0)), kw.get("relation", "at least"))


def _chk_number_words(text, kw):
    return _compare(_count_words(text), int(kw.get("num_words", 0)), kw.get("relation", "at least"))


def _chk_nth_paragraph_first_word(text, kw):
    nth = int(kw.get("nth_paragraph", 1))
    first = str(kw.get("first_word", "")).strip().lower()
    paras = [p.strip() for p in re.split(r"\n\s*\n", str(text or "")) if p.strip()]
    need = int(kw.get("num_paragraphs", 0) or 0)
    if need and len(paras) != need: return False
    if len(paras) < nth: return False
    tokens = _word_tokens(paras[nth - 1])
    return bool(tokens) and tokens[0].lower() == first


def _chk_no_comma(text, kw):
    return ("," not in str(text or "")) and ("，" not in str(text or ""))


def _chk_end_phrase(text, kw):
    phrase = str(kw.get("end_phrase", ""))
    return bool(phrase) and str(text or "").strip().endswith(phrase)


def _chk_quotation(text, kw):
    s = str(text or "").strip()
    return len(s) >= 2 and s.startswith('"') and s.endswith('"')


# ===== Strict 指令 id → 判定函数映射 =====
CHECKERS = {
    "change_case:capital_word_frequency": _chk_capital_word_frequency,
    "change_case:english_capital": _chk_capital,
    "change_case:english_lowercase": _chk_lowercase,
    "combination:repeat_prompt": _chk_repeat_prompt,
    "combination:two_responses": _chk_two_responses,
    "detectable_content:number_highlighted_sections": _chk_highlighted_sections,
    "detectable_content:number_placeholders": _chk_number_placeholders,
    "detectable_content:postscript": _chk_postscript,
    "detectable_format:number_highlighted_sections": _chk_highlighted_sections,
    "detectable_format:multiple_sections": _chk_multiple_sections,
    "detectable_format:postscript": _chk_postscript,
    "detectable_format:constrained_response": _chk_constrained_response,
    "detectable_format:json_format": _chk_json_format,
    "detectable_format:number_bullet_lists": _chk_number_bullet_lists,
    "detectable_format:title": _chk_title,
    "keywords:existence": _chk_keywords_existence,
    "keywords:frequency": _chk_keywords_frequency,
    "keywords:forbidden_words": _chk_forbidden_words,
    "keywords:letter_frequency": _chk_letter_frequency,
    "language:response_language": _chk_response_language,
    "length_constraints:number_sentences": _chk_number_sentences,
    "length_constraints:number_words": _chk_number_words,
    "length_constraints:number_paragraphs": _chk_number_paragraphs,
    "length_constraints:nth_paragraph_first_word": _chk_nth_paragraph_first_word,
    "punctuation:no_comma": _chk_no_comma,
    "startend:end_checker": _chk_end_phrase,
    "startend:quotation": _chk_quotation,
}


def verify_strict(text, instruction_ids, kwargs_list):
    """逐条判定 strict，返回 [{id, passed, error}]。passed=None 表示 unsupported / error。"""
    results = []
    for i, iid in enumerate(instruction_ids):
        kw = kwargs_list[i] if i < len(kwargs_list) and isinstance(kwargs_list[i], dict) else {}
        fn = CHECKERS.get(iid)
        if fn is None:
            results.append({"id": iid, "passed": None, "error": "unsupported"})
            continue
        try:
            passed = fn(text, kw)
        except Exception as e:
            results.append({"id": iid, "passed": None, "error": f"{type(e).__name__}: {e}"})
            continue
        results.append({"id": iid, "passed": None if passed is None else bool(passed), "error": None})
    return results


def _tally(per_instruction):
    vals = [x["passed"] for x in per_instruction if x["passed"] is not None]
    all_pass = bool(all(vals)) if vals else None
    return sum(vals), len(vals), all_pass

## 5. Loose 判分器

论文用的是 **instruction-level loose accuracy**（第 3.1 节原文："allows minor formatting deviations (e.g., markdown wrappers)"）。

Loose 和 Strict 的核心差异：对**被检查文本做宽松归一化**后再送进 strict 判定器。这里采用三层预处理：

1. **剥离 markdown 包裹**：去掉 ```json / ``` / ```python 代码块、`xxx` 内联代码、--- / *** 分隔线
2. **归一化空白**：把所有连续空白折叠成单个空格，去掉首尾空白与句末标点
3. **针对特定约束的额外宽容**：`json_format` 允许在全文中搜索 JSON 对象，而不要求全文即 JSON；`quotation` 允许引号外包裹 markdown

In [ ]:
# ===== Loose 预处理：对被检查文本做宽松归一化 =====
def _preprocess_loose(text):
    """
    把 strict 要检查的文本做宽松归一化，再送回 strict 判定器。
    这样只写一套判定逻辑，loose 只是换了输入。
    """
    s = str(text or "")

    # 1. 剥离 fenced code blocks（```lang ... ```）
    s = re.sub(r"^```[A-Za-z0-9_+\-]*\s*\n?", "", s, flags=re.MULTILINE)
    s = re.sub(r"\n?```\s*$", "", s, flags=re.MULTILINE)

    # 2. 剥离行内代码 ``xxx``
    s = re.sub(r"``([^`]+)``", r"\1", s)
    s = re.sub(r"`([^`]+)`", r"\1", s)

    # 3. 剥离常见 markdown 装饰：**加粗**、*斜体*、~~删除线~~
    s = re.sub(r"\*\*([^*]+)\*\*", r"\1", s)
    s = re.sub(r"~~([^~]+)~~", r"\1", s)

    # 4. 把列表前缀 - / * / 1. 等从行首去掉
    s = re.sub(r"(?m)^\s*[-*+]\s+", "", s)
    s = re.sub(r"(?m)^\s*\d+\.\s+", "", s)

    # 5. 折叠所有空白（含换行）为单个空格
    s = re.sub(r"\s+", " ", s).strip()

    return s


def verify_loose(text, instruction_ids, kwargs_list):
    """用宽松归一化后的文本跑 strict 判定器。"""
    return verify_strict(_preprocess_loose(text), instruction_ids, kwargs_list)

## 6. 判分 & 汇总

两种条件各产出 541 条的 strict + loose 两套判分。

核心指标：
- **prompt-level accuracy**：一题的所有约束全过才算对（整体正确率，strict 与 loose 各一个）
- **instruction-level accuracy**：所有（题 × 约束）里通过的比例（单条约束正确率，论文 Table 1 报告的就是这个）

In [ ]:
# ===== 判分：DA / CoT 各跑 strict + loose =====
def score_condition(rows, condition_name):
    """对一条件的全部行做 strict + loose 判分。"""
    results = []
    for row in rows:
        ans = row["answer_segment"]  # 判的是最终回答，不是全文（与论文一致）
        p_strict = verify_strict(ans, row["instruction_ids"], row["instruction_kwargs"])
        p_loose = verify_loose(ans, row["instruction_ids"], row["instruction_kwargs"])
        n_pass_s, n_chk_s, all_s = _tally(p_strict)
        n_pass_l, n_chk_l, all_l = _tally(p_loose)
        results.append({
            "uid": row["uid"],
            "k": row["k"],
            "all_pass_strict": all_s,
            "n_pass_strict": n_pass_s,
            "n_check_strict": n_chk_s,
            "all_pass_loose": all_l,
            "n_pass_loose": n_pass_l,
            "n_check_loose": n_chk_l,
            "per_instruction_strict": p_strict,
            "per_instruction_loose": p_loose,
            "n_tokens": row["n_tokens"],
            "raw_output": row["raw_output"],
            "think_segment": row["think_segment"],
            "answer_segment": row["answer_segment"],
            "prompt": row["prompt"],
            "condition": condition_name,
        })
    return results


da_scored = score_condition(da_rows, "DA")
cot_scored = score_condition(cot_rows, "CoT")

# 落盘
for name, scored in [("da", da_scored), ("cot", cot_scored)]:
    with open(RESULT_DIR / f"{name}_scored.jsonl", "w", encoding="utf-8") as f:
        for row in scored:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
print("判分完成，已落盘 da_scored.jsonl / cot_scored.jsonl")

In [ ]:
# ===== 汇总对比 =====
def summarize(scored, label):
    """从 scored 列表算 prompt-level 和 instruction-level 两套指标。"""
    df_s = pd.DataFrame(scored)

    # prompt-level：一题全过才算对
    p_s_vals = [r["all_pass_strict"] for r in df_s["all_pass_strict"] if r is not None]
    p_l_vals = [r["all_pass_loose"] for r in df_s["all_pass_loose"] if r is not None]
    prompt_strict = sum(p_s_vals) / len(p_s_vals) if p_s_vals else float("nan")
    prompt_loose = sum(p_l_vals) / len(p_l_vals) if p_l_vals else float("nan")

    # instruction-level：所有（题 × 约束）通过比例
    tot_pass_s = sum(df_s["n_pass_strict"])
    tot_chk_s = sum(df_s["n_check_strict"])
    tot_pass_l = sum(df_s["n_pass_loose"])
    tot_chk_l = sum(df_s["n_check_loose"])
    inst_strict = tot_pass_s / tot_chk_s if tot_chk_s else float("nan")
    inst_loose = tot_pass_l / tot_chk_l if tot_chk_l else float("nan")

    avg_tok = df_s["n_tokens"].mean()

    return {
        "prompt_strict": prompt_strict,
        "prompt_loose": prompt_loose,
        "instruction_strict": inst_strict,
        "instruction_loose": inst_loose,   # ← 论文 Table 1 报告的就是这一列
        "avg_tokens": avg_tok,
        "n_prompts": len(df_s),
    }


da_summary = summarize(da_scored, "DA")
cot_summary = summarize(cot_scored, "CoT")

print("\n" + "=" * 60)
print(f"{'指标':<22} {'DA':>10}  {'CoT':>10}  {'Δ (DA−CoT)':>12}")
print("-" * 60)
for key, label in [
    ("prompt_strict",    "prompt-level strict"),
    ("prompt_loose",     "prompt-level loose"),
    ("instruction_strict","instruction-level strict"),
    ("instruction_loose", "instruction-level loose ← 论文指标"),
    ("avg_tokens",       "平均生成 tokens"),
]:
    dv = da_summary[key]
    cv = cot_summary[key]
    if key.startswith("avg"):
        print(f"{label:<22} {dv:>10.1f}  {cv:>10.1f}  {dv - cv:>+12.1f}")
    else:
        print(f"{label:<22} {dv:>10.2%}  {cv:>10.2%}  {dv - cv:>+12.2%}")

print("=" * 60)
print(f"论文 Meta-Llama-3-8B-Instruct（Table 1）：DA=75.2%，CoT=59.0%，Δ=+16.2pp")
print("（你这边跑出来的 Δ 如果接近 16pp 就说明复现成功；符号相反意味着 CoT 帮倒忙，与论文一致）")

# 小 dataframe 方便查看
summary_df = pd.DataFrame({
    "DA": [da_summary[k] for k in ["prompt_strict", "prompt_loose", "instruction_strict", "instruction_loose", "avg_tokens"]],
    "CoT": [cot_summary[k] for k in ["prompt_strict", "prompt_loose", "instruction_strict", "instruction_loose", "avg_tokens"]],
    "Δ": [da_summary[k] - cot_summary[k] for k in ["prompt_strict", "prompt_loose", "instruction_strict", "instruction_loose", "avg_tokens"]],
}, index=["prompt_strict", "prompt_loose", "inst_strict", "inst_loose", "avg_tokens"])
display(summary_df)

In [ ]:
# ===== 按约束条数 k 分层查看（可选，看 DA 与 CoT 的 gap 是否随 k 增大）=====
def by_k(scored):
    df_s = pd.DataFrame(scored)
    rows = []
    for k, g in df_s.groupby("k"):
        tot_p_l = sum(g["n_pass_loose"])
        tot_c_l = sum(g["n_check_loose"])
        rows.append({"k": k, "n": len(g), "inst_loose": tot_p_l / tot_c_l})
    return pd.DataFrame(rows)

bk_da = by_k(da_scored).rename(columns={"inst_loose": "DA"})
bk_cot = by_k(cot_scored).rename(columns={"inst_loose": "CoT"})
merged = bk_da.merge(bk_cot, on="k")
merged["Δ (DA−CoT)"] = merged["DA"] - merged["CoT"]
display(merged.to_string(index=False))